# ReSnap v5a: test eksternal rimpang (sekali jalan)

Holdout rimpang di run v5a berasal dari sumber yang sama dengan data train (soda, rimpang10), sehingga angkanya bisa terlalu optimis.
Notebook ini menilai v4b dan v5a pada **test eksternal**: 314 foto dari dataset `rival` yang tidak pernah dipakai untuk training,
pemilihan model, maupun penyaringan ambang. Sebagian besar foto rival berisi rempah lain (lada, cengkeh, adas, kapulaga, dll.) tanpa label,
jadi notebook ini juga mengukur **deteksi palsu** pada foto yang tidak berisi jahe/kunyit/lengkuas.

Tidak ada keputusan training yang diubah berdasarkan hasil ini; angka ini dilaporkan apa adanya.

Kaggle: GPU T4, Internet on. Input: output notebook **ReSnap v5a - Data Train** (berisi `best_v5a.pt` dan `dataset_v5a.zip`).
Waktu: sekitar 10 menit.

In [ ]:
%pip -q install ultralytics==8.3.228

In [ ]:
import json, math, random, shutil, subprocess, sys, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps

assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU T4.'
DEVICE = '0'
INPUT, OUTPUT, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/resnap')
OUT = OUTPUT / 'test_eksternal'
for p in (OUT, TMP):
    p.mkdir(parents=True, exist_ok=True)

REPO = TMP / 'resnap-cv'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AffanAbya30/resnap-cv.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from evaluate_export import evaluate

V4B = REPO / 'models/best.pt'
assert dv.sha256_file(V4B) == '1410f11d9c2cdfad074446d1742ed5ec24d0269122bc51ec5ddde69bafc66809'
V5A = next(INPUT.rglob('best_v5a.pt'), None)
assert V5A, 'best_v5a.pt tidak ditemukan. Add Input: output notebook "ReSnap v5a - Data Train".'
print('v5a:', V5A, dv.sha256_file(V5A)[:12])

EXT = next((p for p in INPUT.rglob('test_eksternal_rimpang') if (p / 'images').is_dir()), None)
if EXT is None:
    z = next(INPUT.rglob('dataset_v5a.zip'), None)
    assert z, 'dataset_v5a.zip tidak ditemukan'
    with zipfile.ZipFile(z) as f:
        members = [m for m in f.namelist() if m.startswith('test_eksternal_rimpang/')]
        f.extractall(TMP / 'd5', members)
    EXT = TMP / 'd5' / 'test_eksternal_rimpang'
imgs = sorted(p for p in (EXT / 'images').iterdir() if p.suffix.lower() in dv.IMG_EXT)
print('Test eksternal:', len(imgs), 'foto')
Y = dv.write_yaml(TMP / 'ext.yaml', EXT, 'images', 'images')
NAMES = dict(enumerate(dv.CLASS_NAMES))
RIMPANG = {NAMES[i]: i for i in (21, 22, 23)}
assert set(RIMPANG) == {'ginger', 'turmeric', 'galangal'}

## 1. mAP pada foto yang berlabel jahe/kunyit/lengkuas

In [ ]:
MODELS = {'v4b': V4B, 'v5a': V5A}
E = {n: evaluate(str(w), Y, 'val', OUT, f'{n}_eksternal', DEVICE) for n, w in MODELS.items()}
overall = pd.DataFrame(E).loc[['images', 'precision', 'recall', 'f1', 'mAP50', 'mAP50_95']]
display(overall.style.format(precision=3).set_caption('Test eksternal rimpang (rival)'))
pc = {n: pd.read_csv(OUT / f'{n}_eksternal' / 'per_class.csv').set_index('class_name') for n in MODELS}
per = pd.DataFrame({f'AP50_{n}': pc[n].reindex(list(RIMPANG))['AP50'] for n in MODELS})
per['objek'] = pc['v5a'].reindex(list(RIMPANG))['instances']
per['delta_poin'] = 100 * (per.AP50_v5a - per.AP50_v4b)
display(per.style.format(precision=3).set_caption('AP50 per kelas'))

## 2. Deteksi palsu: foto tanpa jahe/kunyit/lengkuas (rempah lain) dan salah kelas antar-rimpang

In [ ]:
def labels(p):
    lbl = EXT / 'labels' / f'{p.stem}.txt'
    out = []
    if lbl.exists():
        for line in lbl.read_text().split('\n'):
            v = line.split()
            if len(v) >= 5:
                c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
                out.append((c, cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2))
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i + 1e-9)

rows = []
for n, w in MODELS.items():
    from ultralytics import YOLO
    m = YOLO(str(w))
    results = (r for i in range(0, len(imgs), 16)   # per 16 foto supaya memori GPU cukup
               for r in m.predict([str(q) for q in imgs[i:i + 16]], conf=0.5, iou=0.7, imgsz=640,
                                  device=DEVICE, verbose=False, stream=True))
    for r in results:
        p = Path(r.path); gt = labels(p)
        b = r.boxes
        preds = list(zip(b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxyn.cpu().numpy()))
        neg = not gt
        for c, s, box in preds:
            best = max(((iou(box, g[1:]), g[0]) for g in gt), default=(0, None))
            if best[0] >= 0.5 and best[1] == c:
                kind = 'benar'
            elif best[0] >= 0.5:
                kind = f'salah kelas ({NAMES[best[1]]} -> {NAMES[c]})'
            else:
                kind = 'palsu di foto rempah lain' if neg else 'palsu (lokasi)'
            rows.append(dict(model=n, img=p.name, negatif=neg, pred=NAMES[c], conf=float(s), jenis=kind))
    del m; torch.cuda.empty_cache()
det = pd.DataFrame(rows)
n_neg = sum(1 for p in imgs if not labels(p))
print('Foto tanpa label rimpang (rempah lain):', n_neg)
fp_neg = det[det.negatif].groupby('model').img.nunique().reindex(MODELS, fill_value=0)
summary_fp = pd.DataFrame({'foto_negatif_dengan_deteksi': fp_neg,
                           'persen': (100 * fp_neg / max(1, n_neg)).round(1),
                           'deteksi_palsu_total': det[det.negatif].groupby('model').size().reindex(MODELS, fill_value=0)})
display(summary_fp.style.set_caption('Deteksi (conf >= 0.5) pada foto rempah lain: makin kecil makin baik'))
display(det[det.negatif].groupby(['model', 'pred']).size().unstack(0, fill_value=0).sort_values('v5a', ascending=False).head(10)
        .style.set_caption('Kelas yang muncul palsu di foto rempah lain'))
display(det[det.jenis.str.startswith('salah')].groupby(['model', 'jenis']).size().unstack(0, fill_value=0)
        .style.set_caption('Salah kelas pada objek rimpang'))
det.to_csv(OUT / 'deteksi_conf05.csv', index=False)

## 3. Contoh visual dan paket hasil

In [ ]:
random.seed(0)
pos = [p for p in imgs if labels(p)]; neg = [p for p in imgs if not labels(p)]
pick = random.sample(pos, min(8, len(pos))) + random.sample(neg, min(4, len(neg)))
models = {n: YOLO(str(w)) for n, w in MODELS.items()}
tiles = []
for p in pick:
    for n, m in models.items():
        r = m.predict(str(p), conf=0.5, imgsz=640, device=DEVICE, verbose=False)[0]
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); W, H = im.size
        d = ImageDraw.Draw(im); lw = max(2, W // 150)
        for c, x1, y1, x2, y2 in labels(p):
            d.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], outline=(220, 30, 30), width=lw)
        for c, s, (x1, y1, x2, y2) in zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy(), r.boxes.xyxyn.cpu().numpy()):
            d.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], outline=(30, 180, 60), width=lw)
            d.text((x1 * W + 3, y1 * H + 3), f'{NAMES[c]} {s:.2f}', fill=(30, 180, 60))
        im.thumbnail((300, 280)); t = Image.new('RGB', (300, 300), 'white'); t.paste(im, (0, 20))
        ImageDraw.Draw(t).text((4, 4), f'{n} | {"rimpang" if labels(p) else "rempah lain"}', fill='black'); tiles.append(t)
sheet = Image.new('RGB', (6 * 300, math.ceil(len(tiles) / 6) * 300), 'white')
for i, t in enumerate(tiles):
    sheet.paste(t, ((i % 6) * 300, (i // 6) * 300))
sheet.save(OUT / 'contoh_v4b_vs_v5a.jpg', quality=82)

res = {'mAP': {n: {k: (round(float(v), 4) if isinstance(v, (int, float)) else v) for k, v in E[n].items()} for n in MODELS},
       'ap50_per_kelas': per.round(4).to_dict(orient='index'),
       'deteksi_palsu_foto_rempah_lain': summary_fp.to_dict(orient='index'), 'foto_rempah_lain': n_neg}
(OUT / 'ringkasan_test_eksternal.json').write_text(json.dumps(res, indent=2, ensure_ascii=False))
zp = OUTPUT / 'test_eksternal_v5a.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file() and f.suffix in ('.json', '.csv', '.jpg', '.png') and 'val_batch' not in f.name:
            z.write(f, f.relative_to(OUT))
print(json.dumps(res, indent=1, ensure_ascii=False))
print(zp, round(zp.stat().st_size / 1e6, 2), 'MB')